In [2]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("info.txt", encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'info.txt'}, page_content='10th class Marks\n\n**Board of Secondary Education\nTelangana State, India**\n\n**SECONDARY SCHOOL CERTIFICATE**\n**REGULAR** PC/29/4222/04/256517/3\n**TS-EE 524495**\n\n---\n\n**CERTIFIED THAT**\n**KATTA SAI PRANAV REDDY**\n**Father\'s Name:** KATTA SRINIVAS REDDY\n**Mother\'s Name:** KATTA UMARANI\n**Roll No.:** 1929100642\n**Date of Birth:** 03/06/2003 (Zero Three June Two Zero Zero Three)\n**School:** EKALAVYA FOUNDATION SCL NALGONDA, NALGONDA DISTRICT\n**Medium:** ENGLISH\n\nHas appeared and **PASSED SSC EXAMINATION** held in **MARCH–2019**\n\n\n### **The Candidate Secured the Following Grade and Grade Points in Curricular Areas:**\n\n| Subject                  | Grade FA | Grade SA | Overall Grade | Grade Point |\n| ------------------------ | -------- | -------- | ------------- | ----------- |\n| First Language (TELUGU)  | A1       | A1       | A1            | 10          |\n| Third Language (ENGLISH) | A1       | A2       

In [3]:
len(documents)

1

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=500)
docs = text_splitter.split_documents(documents)
len(docs)

46

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L3-v2"
)

c:\Users\saipr\RAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 3907.96it/s]


In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [10]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")


In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",streaming=True
)

result=llm.invoke("Who is virat kohli")

In [13]:
result.content

"Virat Kohli is a renowned Indian international cricketer who is widely regarded as one of the greatest batsmen of all time. He was born on November 5, 1988, in Delhi, India.\n\nKohli has had an illustrious career, playing for the Indian national team in all three formats of the game: Test, One-Day International (ODI), and Twenty20 International (T20I). He has been the captain of the Indian team in all three formats and has led the team to numerous victories.\n\nSome of his notable achievements include:\n\n1. **Fastest to 10,000 ODI runs**: Kohli achieved this milestone in just 205 innings, surpassing the previous record held by Sachin Tendulkar.\n2. **Most runs in a calendar year**: Kohli has held this record multiple times, with his best being 2,818 runs in 2016.\n3. **Highest average in T20Is**: Kohli has an impressive average of over 50 in T20Is, making him one of the most consistent performers in the format.\n4. **Most Test centuries as captain**: Kohli has scored 20 Test centurie

In [ ]:
vectorstore.save_local("faiss_index")


In [ ]:
query = "What are my skills?"



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
retrieved_docs = retriever.invoke(query)
print(retrieved_docs)
print(f"Retrieved {len(retrieved_docs)} documents\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n===== Document {i} =====")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()

)


[Document(id='e2f6ea63-2bf5-4263-9e86-509712781052', metadata={'source': 'info.txt'}, page_content='### **Skills**\n\n* **Tools:** MLflow, DVC, Docker, Git, GitHub Actions, AWS (EC2, S3, ECR), FAISS, Pinecone, Hugging Face, LangChain, LangSmith, FastAPI\n* **Programming & Technical Skills:** Python, SQL, HTML, CSS, Scikit-learn, TensorFlow, Keras, Statistics\n* **Data Science & Machine Learning:** Data Preprocessing, EDA, Feature Engineering, Model Training & Evaluation, Hyperparameter Tuning, Clustering, MLOps, Semantic Search, Retrieval-Augmented Generation (RAG), CNN, RNN, GPT, Transformers, Fine-Tuning, Prompt Engineering\n* **Data Visualization & Analysis:** Pandas, NumPy, Matplotlib, Seaborn\n\n---\n\nhobbies section \n\n---\n\n### **Hobbies & Interests**\nHobbies & Interests\n\n* Playing Cricket\n* Watching Football\n* Reading Books\n* Exploring Latest Advancements in Artificial Intelligence\n* Browsing the Internet for Tech & Knowledge Updates\n\n---\n\n### Contact Information\

In [24]:

# 6. Run the query
query = "what projects does pranav reddy have done?"
response = rag_chain.invoke(query)

print("\n--- AI Response ---")
print(response)


--- AI Response ---
According to the context, Pranav Reddy has worked on the following projects:

1. BigBasket SmartCart – AI Assistant for BigBasket Shopping 

Additionally, as a Data Science Intern at Unified Mentor Pvt. Ltd., Pranav Reddy worked on a project where he:

1. Developed and optimized machine learning models to predict employee attrition.


In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",streaming=True
)

result=llm.invoke("Who is virat kohli")

In [2]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")


# Set your API key (replace with your actual key)

# 1. Define the Context-Stuffed System Prompt
system_prompt = """You are an expert AI recruiter assistant representing Katta Sai Pranav Reddy. 
Your exclusive purpose is to answer questions from potential employers, recruiters, and engineers about Pranav's background, technical skills, and projects.

You must strictly adhere to the following rules:
1. Zero Hallucination: Answer questions using ONLY the information provided in the "Candidate Database" below. Do not infer, guess, or invent any skills, metrics, or experiences not explicitly written here.
2. Handle Missing Info Gracefully: If a user asks a question that cannot be answered by the provided data, respond with: "I don't have that specific information in my context, but you can reach out to Pranav directly at kattapranavreddy@gmail.com to discuss it."
3. Tone: Be concise, professional, confident, and highly technical.

--- CANDIDATE DATABASE ---

**Contact & Identity:**
* Name: Katta Sai Pranav Reddy
* Location: Hyderabad, India
* Email: kattapranavreddy@gmail.com
* GitHub: ka1817
* LinkedIn: pranav-reddy-katta
* Summary: AI and ML Engineer with hands-on experience developing end-to-end Machine Learning and Generative AI solutions. Proficient in data preprocessing, exploratory data analysis (EDA), and predictive modeling techniques to drive data-driven decision-making.

**Education:**
* Degree: B.Tech in Artificial Intelligence and Machine Learning
* Institution: Anurag University, Hyderabad, India (09/2021 - 04/2025)
* CGPA: 8.29
* High School: Sri Chaitanya Junior College, Hyderabad, India (06/2019 - 05/2021)
* Focus & Grade: MPC (Maths, Physics, Chemistry) with 98%

**Work Experience:**
* Machine Learning Intern | iNeuron Intelligence Pvt. Ltd. (Remote, 10/2024 - 11/2024)
    * Conducted extensive data preprocessing and EDA on large customer datasets to identify behavioral patterns and high-value segments.
    * Developed and trained ML models for customer segmentation using the K-Means algorithm, achieving a Silhouette Score of 0.82.
    * Delivered actionable recommendations based on statistical analysis for targeted marketing campaigns.
* Data Science Intern | Unified Mentor Pvt. Ltd. (Remote, 09/2024 - 10/2024)
    * Developed and optimized machine learning models to predict employee attrition for proactive retention strategies.
    * Conducted comprehensive data preprocessing, feature engineering, and EDA to identify key turnover factors.
    * Delivered actionable insights via dashboards and presented strategic recommendations to stakeholders.

**Technical Projects:**
* BigBasket SmartCart (AI-Driven Shopping Assistant) (06/2025 - 07/2025)
    * Led development of an AI assistant using RAG for natural language queries and semantic product search with 95% retrieval accuracy.
    * Built a retrieval pipeline using the gte-small model, FAISS indexing, and Cross-Encoder reranking, achieving a relevance score of 0.89.
    * Designed a modular architecture with FastAPI, HTML/CSS, and Docker, reducing response latency to 2 seconds.
    * Implemented CI/CD using GitHub Actions and deployed on AWS EC2, reducing deployment time by 40%.
* Netflix Customer Churn Prediction (End-to-End ML System)
    * Developed a complete ML pipeline predicting customer churn, achieving 99% recall via advanced feature engineering and hyperparameter tuning.
    * Performed in-depth EDA to identify churn drivers (low engagement, payment methods).
    * Implemented MLOps workflows using DVC, AWS S3, and MLflow.
    * Designed a FastAPI-based REST API, containerized with Docker, and automated deployment via GitHub Actions to AWS EC2.

**Technical Skills:**
* Tools: MLflow, DVC, Docker, Git, GitHub Actions, AWS (EC2, S3, ECR), FAISS, Pinecone, Hugging Face, LangChain, LangSmith, FastAPI.
* Programming & Core Tech: Python, SQL, HTML, CSS, Scikit-learn, TensorFlow, Keras, Statistics.
* ML Domains: Data Preprocessing, EDA, Feature Engineering, Model Training & Evaluation, Hyperparameter Tuning, Clustering, MLOps, Semantic Search, RAG, CNN, RNN, GPT, Transformers, Fine-Tuning, Prompt Engineering.
* Data Visualization: Pandas, NumPy, Matplotlib, Seaborn.
--- END CANDIDATE DATABASE ---"""

# 2. Create the Chat Prompt Template
# This automatically maps the system prompt and the user's incoming question
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

# 3. Initialize the Language Model
# Setting temperature=0 ensures deterministic, factual responses based strictly on your resume
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",streaming=True
)

# 4. Build the Chain using LCEL (LangChain Expression Language)
# Pipeline: Prompt -> LLM -> String Output
resume_chain = prompt_template | llm | StrOutputParser()

# 5. Create an interactive loop to test it

In [4]:
user_input="who pranav ?"
response = resume_chain.invoke({"question": user_input})
        
print(f"\nAssistant: {response}\n")
print("-" * 50)


Assistant: Katta Sai Pranav Reddy is an AI and ML Engineer with hands-on experience in developing end-to-end Machine Learning and Generative AI solutions. He has a strong background in data preprocessing, exploratory data analysis, and predictive modeling techniques to drive data-driven decision-making. You can find more information about his education, work experience, technical projects, and skills in the provided database. If you need more specific details, feel free to ask.

--------------------------------------------------
